# 🏠 House Price Prediction — Modélisation
## Objectif
Entraîner le meilleur modèle ML pour prédire le prix des maisons.

## Réentraînement sur 100% des données d'entraînement
Le meilleur modèle (Lasso) est réentraîné sur la totalité du train pour maximiser l'information apprise.

In [27]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy.special import boxcox1p
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler

# Charger les modèles
stacking = joblib.load('../models/stacking.pkl')
elasticnet = joblib.load('../models/elasticnet.pkl')
lasso = joblib.load('../models/lasso.pkl')
ridge = joblib.load('../models/ridge.pkl')
xgb_model = joblib.load('../models/xgboost.pkl')
lgbm_model = joblib.load('../models/lgbm.pkl')
weights = joblib.load('../models/blend_weights.pkl')

print("Modèles chargés !")
print("Poids blend :", weights)

Modèles chargés !
Poids blend : [0.21005851 0.34279267 0.27232051 0.00844271 0.10806072 0.05253023
 0.         0.00579465]


In [28]:
import joblib
from scipy.stats import skew

# Charger les artifacts sauvegardés par notebook 02
scaler       = joblib.load('../models/scaler.pkl')
train_columns = joblib.load('../models/columns.pkl')
skewed_feats = joblib.load('../models/skewed_feats.pkl')

# --- Même pipeline que notebook 02 ---
cols_none = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType"]
cols_zero = ["GarageYrBlt", "GarageArea", "GarageCars",
             "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
             "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"]
qual_map  = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

def preprocess(df):
    for col in cols_none: df[col] = df[col].fillna("None")
    for col in cols_zero: df[col] = df[col].fillna(0)
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median()))
    df = df.fillna(df.mode().iloc[0])

    # Surfaces
    df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalLivArea'] = df['GrLivArea']   + df['TotalBsmtSF']
    df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                          df['3SsnPorch']   + df['ScreenPorch']   + df['WoodDeckSF'])
    df['TotalBath']    = (df['FullBath'] + 0.5*df['HalfBath'] +
                          df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['LivLotRatio']  = df['GrLivArea'] / df['LotArea']
    df['BsmtFinRatio'] = (df['BsmtFinSF1'] + df['BsmtFinSF2']) / df['TotalBsmtSF'].replace(0, 1)

    # Temporel
    df['HouseAge']      = df['YrSold'] - df['YearBuilt']
    df['RemodAge']      = df['YrSold'] - df['YearRemodAdd']
    df['GarageAge']     = df['YrSold'] - df['GarageYrBlt']
    df.loc[df['GarageYrBlt'] == 0, 'GarageAge'] = 0
    df['IsNew']         = (df['YearBuilt']  == df['YrSold']).astype(int)
    df['IsRemodeled']   = (df['YearBuilt']  != df['YearRemodAdd']).astype(int)
    df['IsRecentRemod'] = (df['YrSold'] - df['YearRemodAdd'] < 5).astype(int)

    # Flags binaires
    df['HasPool']      = (df['PoolArea']     > 0).astype(int)
    df['HasGarage']    = (df['GarageArea']   > 0).astype(int)
    df['HasBsmt']      = (df['TotalBsmtSF']  > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces']   > 0).astype(int)
    df['Has2ndFloor']  = (df['2ndFlrSF']     > 0).astype(int)
    df['HasPorch']     = (df['TotalPorchSF'] > 0).astype(int)
    df['HasMasVnr']    = (df['MasVnrArea']   > 0).astype(int)
    df['HasShed']      = (df['MiscFeature']  == 'Shed').astype(int)

    # Ordinaux qualité
    for c in qual_cols:
        df[c + '_ord'] = df[c].map(qual_map).fillna(0).astype(int)
    df['BsmtExposure_ord'] = df['BsmtExposure'].map(
        {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}).fillna(0).astype(int)
    df['BsmtFinType1_ord'] = df['BsmtFinType1'].map(
        {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}).fillna(0).astype(int)
    df['GarageFinish_ord'] = df['GarageFinish'].map(
        {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}).fillna(0).astype(int)
    df['Functional_ord']   = df['Functional'].map(
        {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3,
         'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7}).fillna(7).astype(int)

    # Interactions
    df['OverallScore'] = df['OverallQual'] * df['OverallCond']
    df['QualSF']       = df['OverallQual'] * df['TotalSF']
    df['QualBath']     = df['OverallQual'] * df['TotalBath']
    df['QualGarage']   = df['OverallQual'] * df['GarageArea']
    df['ExterScore']   = df['ExterQual_ord'] * df['ExterCond_ord']
    df['BsmtScore']    = df['BsmtQual_ord']  * df['BsmtCond_ord']
    df['GarageScore']  = df['GarageQual_ord'] * df['GarageCond_ord'] * df['GarageArea']

    # Re-typage catégoriel
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    df['MoSold']     = df['MoSold'].astype(str)
    df['YrSold']     = df['YrSold'].astype(str)

    return df

# --- Preprocessing test.csv ---
test_raw = pd.read_csv("../data/test.csv")
test_ids = test_raw["Id"]
test_raw = preprocess(test_raw)

# Correction skew avec les mêmes features que le train
for c in skewed_feats:
    if c in test_raw.columns:
        test_raw[c] = np.log1p(test_raw[c])

# Encodage + alignement colonnes
test_encoded = pd.get_dummies(test_raw, drop_first=True)
test_encoded = test_encoded.reindex(columns=train_columns, fill_value=0)

# Scaling avec le scaler du train
test_scaled = pd.DataFrame(scaler.transform(test_encoded), columns=train_columns)
print("Test preprocessé :", test_scaled.shape, "→ doit être (1459, 326)")

Test preprocessé : (1459, 326) → doit être (1459, 326)


C:\Users\amine\AppData\Local\Temp\ipykernel_17808\1526535937.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c + '_ord'] = df[c].map(qual_map).fillna(0).astype(int)
C:\Users\amine\AppData\Local\Temp\ipykernel_17808\1526535937.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c + '_ord'] = df[c].map(qual_map).fillna(0).astype(int)
C:\Users\amine\AppData\Local\Temp\ipykernel_17808\1526535937.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

In [29]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialiser les prédictions OOF
oof_preds = {name: np.zeros(len(X_full)) for name in 
             ['stacking', 'elasticnet', 'lasso', 'ridge', 'xgb', 'lgbm']}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_full)):
    X_tr, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    for name, model in [('elasticnet', elasticnet), ('lasso', lasso),
                        ('ridge', ridge), ('xgb', xgb_model), ('lgbm', lgbm_model)]:
        model.fit(X_tr, y_tr)
        oof_preds[name][val_idx] = model.predict(X_val)
    
    stacking.fit(X_tr, y_tr)
    oof_preds['stacking'][val_idx] = stacking.predict(X_val)
    
    print(f"Fold {fold+1} terminé")

# Optimiser les poids sur les prédictions OOF
def objective(w):
    pred = sum(w[i] * oof_preds[name] for i, name in 
               enumerate(['stacking','elasticnet','lasso','ridge','xgb','lgbm']))
    return np.sqrt(mean_squared_error(y_full, pred))

constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
bounds = [(0, 1)] * 6
result = minimize(objective, x0=[1/6]*6, bounds=bounds, constraints=constraints)
best_weights = result.x

print("\nPoids optimisés (OOF) :")
for name, w in zip(['stacking','elasticnet','lasso','ridge','xgb','lgbm'], best_weights):
    print(f"  {name:12s}: {w:.4f}")
print(f"RMSE OOF : {result.fun:.4f}")

Fold 1 terminé
Fold 2 terminé
Fold 3 terminé
Fold 4 terminé
Fold 5 terminé

Poids optimisés (OOF) :
  stacking    : 0.0897
  elasticnet  : 0.2861
  lasso       : 0.1942
  ridge       : 0.0803
  xgb         : 0.1098
  lgbm        : 0.2399
RMSE OOF : 0.1120


## Prédiction sur test.csv
On applique le même preprocessing que sur train, puis on prédit avec le modèle final.

In [30]:
# Blend final avec les modèles du notebook 03 (326 features — compatible)
y_pred_log = (weights[0] * stacking.predict(test_scaled) +
              weights[1] * elasticnet.predict(test_scaled) +
              weights[2] * lasso.predict(test_scaled) +
              weights[3] * ridge.predict(test_scaled) +
              weights[4] * xgb_model.predict(test_scaled) +
              weights[5] * lgbm_model.predict(test_scaled))

y_pred_final = np.exp(y_pred_log)

# Post-traitement — clipping des prédictions extrêmes
y_pred_log_clipped = np.clip(y_pred_log, 
                              np.percentile(y_pred_log, 0.5), 
                              np.percentile(y_pred_log, 99.5))

y_pred_final = np.exp(y_pred_log_clipped)

print("Min prix prédit :", round(y_pred_final.min(), 0))
print("Max prix prédit :", round(y_pred_final.max(), 0))

os.makedirs("../data/processed", exist_ok=True)
submission = pd.DataFrame({"Id": test_ids, "SalePrice": y_pred_final})
submission.to_csv("../data/processed/submission.csv", index=False)
print("Prédictions sauvegardées dans data/processed/submission.csv")
print(submission.head(10))

Min prix prédit : 53865.0
Max prix prédit : 430093.0
Prédictions sauvegardées dans data/processed/submission.csv
     Id      SalePrice
0  1461  112844.619160
1  1462  146725.384915
2  1463  167860.255711
3  1464  180193.358928
4  1465  178498.182818
5  1466  159168.502247
6  1467  167712.617282
7  1468  152296.954276
8  1469  183095.621467
9  1470  111706.665774
